In [4]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


PROJECT_PATH = Path("C:/Users/Архиповы/source/repos/modem")
sys.path.insert(0, str(PROJECT_PATH / "src"))

from modem.blocks import (MessageSource, StringToBits, PCM, BPSK, 
                          DPSK, HammingDecoder, HammongEncoder, GFSKDemodulator, 
                          BFSK, Channel, BFSKDemodulator,
                          Demodulator, BitsToString, GFSK, PreambleSearcher, 
                          DPSKDemodulator, BFSKCoherentDemodulator, 
                          FileSource)
from modem.pipeline import run

MAX_DISPLAY_SAMPLES = 1000
MAX_DISPLAY_BITS = 30

BLE_PREAMBLE = "01010101"
BLE_ACCESS_ADDRESS = "01101011011111011001000101110001"

DEFAULT_FILE_PATH = r"K:\Загрузки\38833FF26BA1D.UnigramPreview_g9c9v27vpyspw!App\rx6677.23067800.dat"

def time_graph(signal, fs, name, ax=None, is_step=False):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(9, 2.5))
    n_show = min(len(signal), MAX_DISPLAY_SAMPLES)
    t = np.arange(n_show) / fs
    if np.iscomplexobj(signal):
        display_signal = np.abs(signal[:n_show])
    else:
        display_signal = signal[:n_show]
    if is_step:
        ax.step(t, display_signal, where='post', color='blue', linewidth=1)
    else:
        ax.plot(t, display_signal, color='blue', linewidth=1)
    ax.grid(True, alpha=0.3)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Время (с)')
    return ax

def spectrum_graph(signal, fs, name, ax=None, max_samples=512):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(9, 2.5))
    n = min(len(signal), max_samples)
    signal_short = signal[:n]
    fft_vals = np.fft.fft(signal_short)
    amp = np.abs(fft_vals) / n
    freq = np.fft.fftfreq(n, 1/fs)
    amp_shift = np.fft.fftshift(amp)
    freq_shift = np.fft.fftshift(freq)
    ax.plot(freq_shift, amp_shift, color='red', linewidth=1)
    ax.grid(True, alpha=0.3)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Частота (Гц)')
    return ax

def calculate_ber(original_bits, recovered_bits):
    if len(original_bits) == 0 or len(recovered_bits) == 0:
        return 1.0
    min_len = min(len(original_bits), len(recovered_bits))
    if min_len == 0:
        return 1.0
    errors = np.sum(original_bits[:min_len] != recovered_bits[:min_len])
    return errors / min_len

def bt_swap_bits(b: int) -> int:
    return int(f'{b:08b}'[::-1], 2)


def ble_packet_data_dewhitening(data: bytes, channel: int) -> bytes:
    lfsr = (bt_swap_bits(channel) | 2) & 0xFF
    output = bytearray(data)
    for i in range(len(output)):
        d = bt_swap_bits(output[i])
        j = 128
        while j >= 1:
            if lfsr & 0x80:
                lfsr ^= 0x11
                d ^= j
            lfsr = (lfsr << 1) & 0xFF
            j >>= 1
        output[i] = bt_swap_bits(d)
    return bytes(output)


# --- ИНТЕРФЕЙС ---
mod_type = widgets.RadioButtons(
    options=['BPSK', 'BFSK', 'DPSK', "GFSK", "BLE из файла"],
    value='BPSK', description='Модуляция:'
)
demod_type = widgets.RadioButtons(
    options=['Когерентный', 'Некогерентный'],
    value='Когерентный', description='BFSK:'
)
fc_slider = widgets.IntSlider(value=8000000, min=100000, max=10000000, step=1000, description='Fc (Гц):')
Rb_slider = widgets.IntSlider(value=1000000, min=10000, max=10000000, step=1000, description='R_b (бит/с):')
h_slider = widgets.FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1, description='h (BFSK):')
k_slider = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.01, description='Затухание:')
snr_slider = widgets.FloatSlider(value=50, min=-20, max=50, step=1, description='SNR (дБ):')
noise_type = widgets.Dropdown(options=['gaussian', 'laplace', 'none'], value='gaussian', description='Шум:')
phase_noise_slider = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description='Фазовый шум:')
use_doppler = widgets.Checkbox(value=False, description='Доплер')
velocity_slider = widgets.FloatSlider(value=30.0, min=0.0, max=10000.0, step=10.0, description='Скорость (м/с):')
angle_slider = widgets.FloatSlider(value=50.0, min=0.0, max=3.14, step=0.1, description='Угол (рад):')
use_hamming = widgets.Checkbox(value=False, description='Код Хэмминга')
source_type = widgets.RadioButtons(
    options=[('Текст', 'text'), ('Файл', 'file')],
    value='text', description='Источник:'
)
text_input = widgets.Textarea(value='Hello DSP!', description='Текст:', layout=widgets.Layout(width='100%', height='80px'))
file_input = widgets.Text(value=DEFAULT_FILE_PATH, description='Файл:', disabled=True, layout=widgets.Layout(width='100%'))

def update_source(*args):
    file_input.disabled = (source_type.value != 'file')
    if source_type.value == 'file':
        mod_type.value = "BLE из файла"
source_type.observe(update_source, 'value')

def update_mod_type(*args):
    if mod_type.value == 'BFSK':
        demod_type.layout.visibility = 'visible'
        h_slider.layout.visibility = 'visible'
    else:
        demod_type.layout.visibility = 'hidden'
        h_slider.layout.visibility = 'hidden'
mod_type.observe(update_mod_type, 'value')

run_button = widgets.Button(description='Запуск', button_style='success', layout=widgets.Layout(width='200px'))

left_panel = widgets.VBox([
    widgets.HTML("<h3>Параметры</h3>"),
    mod_type, demod_type, fc_slider, Rb_slider, h_slider,
    widgets.HTML("<hr>"),
    k_slider, snr_slider, noise_type, phase_noise_slider, 
    use_doppler, velocity_slider, angle_slider, use_hamming,
    widgets.HTML("<hr>"),
    source_type, text_input, file_input,
    widgets.HTML("<hr>"), run_button
], layout=widgets.Layout(width='340px', min_width='340px', border='1px solid gray', padding='10px'))

right_panel = widgets.Output(layout=widgets.Layout(flex='1 1 auto', overflow='auto', padding='10px'))
main_ui = widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width='100%', align_items='flex-start'))
display(main_ui)
update_mod_type()


def process_ble_file(filepath):
    if not Path(filepath).exists():
        print("Файл не найден")
        return
    
    FS = 8000000
    SYM_RATE = 1000000
    SPS = FS // SYM_RATE
    ctx = {}
    
    print("=== Обработка BLE ===")
    source = FileSource(filepath=filepath)
    sig = source.process(None, ctx=ctx)
    print(f"Отсчетов: {len(sig)}")
    
    if len(sig) == 0:
        return
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
    time_graph(np.abs(sig), FS, "Огибающая", ax1)
    spectrum_graph(sig, FS, "Спектр", ax2)
    plt.tight_layout()
    plt.show()
    
    print("\nПоиск преамбулы...")
    searcher = PreambleSearcher(threshold=0.45, sps=SPS)
    result = searcher.process(sig, ctx=ctx)
    
    if not result['found']:
        print("Преамбула не найдена")
        return
    
    start_sample = result['start']
    print(f"Преамбула найдена: отсчет {start_sample}, корреляция {result['correlation']:.4f}")
    
    packet_iq = sig[start_sample:]
    
    print("\nДемодуляция GFSK...")
    demod = GFSKDemodulator(samples_per_symbol=SPS, samp_rate=FS)
    recovered_bits = demod.process(packet_iq, ctx=ctx)
    
    if recovered_bits.dtype in [np.float64, np.float32]:
        working_bits = np.where(recovered_bits > 0, 1, 0).astype(np.int8)
    else:
        working_bits = recovered_bits.astype(np.int8)
    
    print(f"Бит: {len(working_bits)}")
    
    bytes_lsb = []
    for i in range(len(working_bits) // 8):
        byte_bits = working_bits[i*8:(i+1)*8]
        val = sum(int(b) << j for j, b in enumerate(byte_bits))
        bytes_lsb.append(val)
    bytes_lsb = bytes(bytes_lsb)
    
    target = bytes([0xD6, 0xBE, 0x89, 0x8E])
    pos = bytes_lsb.find(target)
    
    if pos >= 0:
        for ch in range(40):
            aa = bytes_lsb[pos:pos+4]
            body = ble_packet_data_dewhitening(bytes_lsb[pos+4:], ch)
            packet = aa + body
            
            if len(packet) < 15:
                continue
            
            h1 = packet[4]
            h2 = packet[5]
            
            if h1 == 0x42 and h2 == 0x06:
                pay_len = h2
                if len(packet) < 6 + pay_len + 3:
                    continue
                
                mac_bytes = packet[6:6+pay_len]
                crc_bytes = packet[6+pay_len:6+pay_len+3]
                
                mac_str = ':'.join(f'{b:02X}' for b in reversed(mac_bytes))
                crc_val = (bt_swap_bits(crc_bytes[0]) << 16) | (bt_swap_bits(crc_bytes[1]) << 8) | bt_swap_bits(crc_bytes[2])
                
                print(f"Access Address: 0x8E89BED6")
                print(f"Канал: {ch}")
                print(f"Заголовок: 0x{h1:02X}{h2:02X}")
                print(f"Длина: {pay_len}")
                print(f"MAC: {mac_str}")
                print(f"CRC: 0x{crc_val:06X}")
                print(f"Пакет: {packet[:15].hex(' ').upper()}")
                break


def run_simulation(b):
    with right_panel:
        clear_output(wait=True)
        mod_type_val = mod_type.value
        source_val = source_type.value
        if source_val == 'file' and "file" not in mod_type_val:
            mod_type_val = "BLE из файла"
            mod_type.value = mod_type_val
        if mod_type_val == "BLE из файла":
            process_ble_file(file_input.value)
            return
        
        fc = fc_slider.value
        R_b = Rb_slider.value
        sps_val = 8
        samp_rate = sps_val * R_b
        text = text_input.value
        
        print("=== СИМУЛЯЦИЯ ===")
        print(f"Несущая: {fc/1e6:.2f} МГц, Битрейт: {R_b/1e3:.1f} кбит/с")
        
        ctx = {}
        capture = {}
        
        blocks = [
            MessageSource(filepath=None, default_text=text or "Hello", name="MS"),
            StringToBits(encoding='utf-8', name="StB"),
        ]
        if use_hamming.value:
            blocks.append(HammongEncoder(name="HE"))
        blocks.append(PCM(name="PCM"))
        
        if mod_type_val == "BPSK":
            blocks.append(BPSK(sps=sps_val, name="bpsk"))
            blocks.append(Channel(k=k_slider.value, snr_db=snr_slider.value, noise_type=noise_type.value,
                                  phase_noise_std=phase_noise_slider.value, use_doppler=use_doppler.value,
                                  velocity=velocity_slider.value, angle=angle_slider.value,
                                  fc=fc, samp_rate=samp_rate, name="channel"))
            blocks.append(Demodulator(sps=sps_val, name="demod"))
        elif mod_type_val == "GFSK":
            blocks.append(GFSK(sps=sps_val, bt=0.5, h=0.5, name="gfsk"))
            blocks.append(Channel(k=k_slider.value, snr_db=snr_slider.value, noise_type=noise_type.value,
                                  phase_noise_std=phase_noise_slider.value, use_doppler=use_doppler.value,
                                  velocity=velocity_slider.value, angle=angle_slider.value,
                                  fc=fc, samp_rate=samp_rate, name="channel"))
            blocks.append(GFSKDemodulator(samples_per_symbol=sps_val, samp_rate=samp_rate, name="gfsk_demod"))
        elif mod_type_val == "DPSK":
            blocks.append(DPSK(sps=sps_val, name="dpsk"))
            blocks.append(Channel(k=k_slider.value, snr_db=snr_slider.value, noise_type=noise_type.value,
                                  phase_noise_std=phase_noise_slider.value, use_doppler=use_doppler.value,
                                  velocity=velocity_slider.value, angle=angle_slider.value,
                                  fc=fc, samp_rate=samp_rate, name="channel"))
            blocks.append(DPSKDemodulator(sps=sps_val, name="dpsk_demod"))
        else:
            blocks.append(BFSK(sps=sps_val, h=h_slider.value, name="BFSK"))
            blocks.append(Channel(k=k_slider.value, snr_db=snr_slider.value, noise_type=noise_type.value,
                                  phase_noise_std=phase_noise_slider.value, use_doppler=use_doppler.value,
                                  velocity=velocity_slider.value, angle=angle_slider.value,
                                  fc=fc, samp_rate=samp_rate, name="channel"))
            if demod_type.value == "Когерентный":
                blocks.append(BFSKCoherentDemodulator(sps=sps_val, h=h_slider.value, name="BFSKDM"))
            else:
                blocks.append(BFSKDemodulator(sps=sps_val, h=h_slider.value, name="BFSKDM"))
        
        if use_hamming.value:
            blocks.append(HammingDecoder(name="HammDecoder"))
        blocks.append(BitsToString(encoding='utf-8', lsb_first=False, name="BSt"))
        
        try:
            result = run(blocks, x=None, ctx=ctx, capture=capture)
        except Exception as e:
            print(f"Ошибка: {e}")
            import traceback
            traceback.print_exc()
            return
        
        original_text = capture.get("MS", "???")
        decoded_text = capture.get("BSt", "???")
        original_bits = np.array(capture.get("StB", []), dtype=np.int8)
        nrz_signal = np.array(capture.get("PCM", []))
        
        if mod_type_val == "BPSK":
            demod_bits = np.array(ctx.get("bpsk_demod", []))
        elif mod_type_val == "DPSK":
            demod_bits = np.array(ctx.get("dpsk_demod", []))
        elif mod_type_val == "GFSK":
            demod_bits = np.array(ctx.get("gfsk_demod", []))
        else:
            demod_bits = np.array(ctx.get("bfsk_demod", []))
        
        if use_hamming.value:
            recovered_bits_float = np.array(ctx.get("after_hamming", []))
        else:
            recovered_bits_float = demod_bits
        
        if len(recovered_bits_float) > 0:
            if recovered_bits_float.dtype in [np.float64, np.float32]:
                recovered_bits = np.where(recovered_bits_float > 0, 1, 0).astype(np.int8)
            else:
                recovered_bits = recovered_bits_float.astype(np.int8)
        else:
            recovered_bits = np.array([], dtype=np.int8)
        
        min_len = min(len(original_bits), len(recovered_bits))
        ber = calculate_ber(original_bits[:min_len], recovered_bits[:min_len]) if min_len > 0 else 1.0
        
        print(f"\n=== РЕЗУЛЬТАТЫ ===")
        print(f"Отправлено: {original_text}")
        print(f"Получено: {decoded_text}")
        print(f"BER: {ber:.8f}")
        
        # График исходных битов
        if len(original_bits) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(original_bits.astype(float), 1, "Исходные биты", ax1, is_step=True)
            spectrum_graph(original_bits.astype(float), 1, "Спектр битов", ax2)
            plt.tight_layout()
            plt.show()
        
        # График NRZ сигнала
        if len(nrz_signal) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(nrz_signal, 1, "NRZ сигнал", ax1, is_step=True)
            spectrum_graph(nrz_signal, 1, "Спектр NRZ", ax2)
            plt.tight_layout()
            plt.show()
        
        # График принятых битов
        if len(recovered_bits) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(recovered_bits.astype(float), 1, "Принятые биты", ax1, is_step=True)
            spectrum_graph(recovered_bits.astype(float), 1, "Спектр принятых битов", ax2)
            plt.tight_layout()
            plt.show()
        
        # Сравнение битов при ошибках
        if ber > 0 and len(original_bits) > 0 and len(recovered_bits) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(9, 2.5))
            min_len = min(len(original_bits), len(recovered_bits))
            n_show = min(min_len, 50)
            ax.step(np.arange(n_show), original_bits[:n_show], where='post', color='blue', linewidth=2, label='Исходные')
            ax.step(np.arange(n_show), recovered_bits[:n_show], where='post', color='red', linewidth=2, linestyle='--', label='Принятые')
            errors = original_bits[:n_show] != recovered_bits[:n_show]
            error_positions = np.where(errors)[0]
            if len(error_positions) > 0:
                ax.scatter(error_positions, [1.1] * len(error_positions), color='red', marker='v', s=50, label='Ошибки')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)
            ax.set_title(f"Сравнение бит (BER={ber:.4f})", fontsize=10)
            ax.set_ylim(-0.2, 1.3)
            plt.tight_layout()
            plt.show()


run_button.on_click(run_simulation)